In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
path1 = "D:\Khushan_BDA\ABD\SparkSession\Data\Road_accidents.csv"
path2 = "D:\Khushan_BDA\ABD\SparkSession\Data\Road_accidents_time.csv"

In [5]:
spark = SparkSession.builder \
    .appName("Road Accidents") \
    .getOrCreate()

In [6]:
df = spark.read.csv(
    path1,
    header=True,
    inferSchema=True
)

In [7]:
time_df = spark.read.csv(
    path2,
    header=True,
    inferSchema=True
)

In [9]:
from functools import reduce

df = df.withColumn(
    'Total',
    reduce(lambda x, y: x + y,
           [col(month) for month in months])
)

In [10]:
df.filter(col('YEAR') == 2013) \
    .groupBy('STATE/UT') \
    .agg(sum('Total').alias('Total_Accidents')) \
    .orderBy(desc('Total_Accidents')) \
    .limit(1) \
    .show()

+----------+---------------+
|  STATE/UT|Total_Accidents|
+----------+---------------+
|Tamil Nadu|          66238|
+----------+---------------+



In [11]:
df.groupBy('STATE/UT') \
    .agg(
        round(avg('Total'), 2).alias('Average_Monthly_Accidents')
    ) \
    .orderBy(desc('Average_Monthly_Accidents')) \
    .show()

+---------------+-------------------------+
|       STATE/UT|Average_Monthly_Accidents|
+---------------+-------------------------+
|     Tamil Nadu|                 60862.36|
|    Maharashtra|                 45829.57|
|      Karnataka|                 41768.64|
| Andhra Pradesh|                 39058.64|
|         Kerala|                  37011.5|
| Madhya Pradesh|                 28143.36|
|      Rajasthan|                 23118.07|
|        Gujarat|                 21451.79|
|  Uttar Pradesh|                 19609.14|
|    West Bengal|                 13526.79|
|        Haryana|                  9422.93|
|     Delhi (Ut)|                  8535.69|
|   Chhattisgarh|                   8415.5|
|       Delhi Ut|                   8310.0|
|         Odisha|                  8231.21|
|          Bihar|                  6617.71|
|Jammu & Kashmir|                  5915.71|
|          Assam|                  4408.43|
|         Punjab|                  3343.14|
|      Jharkhand|               

In [12]:
df.select([
    sum(col(month)).alias(month)
    for month in months
]).show()

+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
|JANUARY|FEBRUARY| MARCH| APRIL|   MAY|  JUNE|  JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
| 482719|  459272|486141|479663|521563|473574|440263|438351|   435302| 454961|  457192|  473905|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+



In [13]:
monthly_totals = []

for month in months:
    value = df.select(sum(col(month)).alias('Total')).collect()[0]['Total']
    monthly_totals.append((month, value))

monthly_totals

[('JANUARY', 482719),
 ('FEBRUARY', 459272),
 ('MARCH', 486141),
 ('APRIL', 479663),
 ('MAY', 521563),
 ('JUNE', 473574),
 ('JULY', 440263),
 ('AUGUST', 438351),
 ('SEPTEMBER', 435302),
 ('OCTOBER', 454961),
 ('NOVEMBER', 457192),
 ('DECEMBER', 473905)]

In [15]:
time_slots = [
    '0-3 hrs. (Night)',
    '3-6 hrs. (Night)',
    '6-9 hrs (Day)',
    '9-12 hrs (Day)',
    '12-15 hrs (Day)',
    '15-18 hrs (Day)',
    '18-21 hrs (Night)',
    '21-24 hrs (Night)'
]

In [17]:
time_df.filter(col('STATE/UT') == 'Karnataka') \
    .select([
        sum(col(slot)).alias(slot)
        for slot in time_slots
    ]) \
    .show()

{"ts": "2026-09-09 16:55:16.261", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `0-3 hrs`.` (Night)` cannot be resolved. Did you mean one of the following? [`0-3 hrs. (Night)`, `3-6 hrs. (Night)`, `18-21 hrs (Night)`, `21-24 hrs (Night)`, `6-9 hrs (Day)`]. SQLSTATE: 42703", "context": {"file": "line 3 in cell [18]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o392.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `0-3 hrs`.` (Night)` cannot be resolved. Did you mean one of the following? [`0-3 hrs. (Night)`, `3-6 hrs. (Night)`, `18-21 hrs (Night)`, `21-24 hrs (Night)`, `6-9 hrs (Day)`]. SQLSTATE: 42703;\n'Aggregate ['sum('0-3 hrs. (Night)) AS 0-3 hrs. (Night)#367, 'sum('3-6 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `0-3 hrs`.` (Night)` cannot be resolved. Did you mean one of the following? [`0-3 hrs. (Night)`, `3-6 hrs. (Night)`, `18-21 hrs (Night)`, `21-24 hrs (Night)`, `6-9 hrs (Day)`]. SQLSTATE: 42703;
'Aggregate ['sum('0-3 hrs. (Night)) AS 0-3 hrs. (Night)#367, 'sum('3-6 hrs. (Night)) AS 3-6 hrs. (Night)#368, sum(6-9 hrs (Day)#52) AS 6-9 hrs (Day)#369L, sum(9-12 hrs (Day)#53) AS 9-12 hrs (Day)#370L, sum(12-15 hrs (Day)#54) AS 12-15 hrs (Day)#371L, sum(15-18 hrs (Day)#55) AS 15-18 hrs (Day)#372L, sum(18-21 hrs (Night)#56) AS 18-21 hrs (Night)#373L, sum(21-24 hrs (Night)#57) AS 21-24 hrs (Night)#374L]
+- Filter (STATE/UT#48 = Karnataka)
   +- Relation [STATE/UT#48,YEAR#49,0-3 hrs. (Night)#50,3-6 hrs. (Night)#51,6-9 hrs (Day)#52,9-12 hrs (Day)#53,12-15 hrs (Day)#54,15-18 hrs (Day)#55,18-21 hrs (Night)#56,21-24 hrs (Night)#57] csv


In [18]:
df.filter(
    (col('YEAR') >= 2001) &
    (col('YEAR') <= 2014)
) \
.groupBy('STATE/UT') \
.agg(
    sum('Total').alias('Total_Accidents')
) \
.orderBy(desc('Total_Accidents')) \
.show()

+---------------+---------------+
|       STATE/UT|Total_Accidents|
+---------------+---------------+
|     Tamil Nadu|         852073|
|    Maharashtra|         641614|
|      Karnataka|         584761|
| Andhra Pradesh|         546821|
|         Kerala|         518161|
| Madhya Pradesh|         394007|
|      Rajasthan|         323653|
|        Gujarat|         300325|
|  Uttar Pradesh|         274528|
|    West Bengal|         189375|
|        Haryana|         131921|
|   Chhattisgarh|         117817|
|         Odisha|         115237|
|     Delhi (Ut)|         110964|
|          Bihar|          92648|
|Jammu & Kashmir|          82820|
|          Assam|          61718|
|         Punjab|          46804|
|      Jharkhand|          46209|
|            Goa|          44931|
+---------------+---------------+
only showing top 20 rows


In [19]:
state_avg = df.filter(
    (col('YEAR') >= 2001) &
    (col('YEAR') <= 2014)
) \
.groupBy('STATE/UT') \
.agg(
    avg('Total').alias('State_Average')
)

In [20]:
accidents_2014 = df.filter(
    col('YEAR') == 2014
).select(
    'STATE/UT',
    col('Total').alias('Accidents_2014')
)

In [21]:
result = accidents_2014.join(
    state_avg,
    on='STATE/UT'
)

result.filter(
    col('Accidents_2014') < col('State_Average')
).select(
    'STATE/UT',
    'Accidents_2014',
    round('State_Average', 2).alias('State_Average')
).show()

+-----------------+--------------+-------------+
|         STATE/UT|Accidents_2014|State_Average|
+-----------------+--------------+-------------+
|         Nagaland|            28|        55.79|
|           Kerala|         35872|      37011.5|
|      Daman & Diu|            39|        45.93|
|  Jammu & Kashmir|          5778|      5915.71|
|       Puducherry|           671|       1513.5|
|Arunachal Pradesh|           185|       242.07|
|           Sikkim|           130|        189.0|
|       Chandigarh|           366|       457.93|
|      Maharashtra|         44382|     45829.57|
|          Tripura|           716|       752.43|
|      Uttarakhand|           801|      1299.43|
+-----------------+--------------+-------------+



In [22]:
df.select([
    sum(col(month)).alias(month)
    for month in months
]).show()

+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
|JANUARY|FEBRUARY| MARCH| APRIL|   MAY|  JUNE|  JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+
| 482719|  459272|486141|479663|521563|473574|440263|438351|   435302| 454961|  457192|  473905|
+-------+--------+------+------+------+------+------+------+---------+-------+--------+--------+

